# Classification Pipeline

---

## Purpose
The purpose of the pipeline is to train and deploy the LinearSVC model for tweet classification, namely seeing if there is a delay or cancellation in a particular train-line. This is done to assist in the creation of the dataset that is used to train the predictive model we are building in the transport microservice. The goal of this pipeline is to have a deployed model that can with extremely high accuracy classify if a train-line is delayed or cancelled given the tweet's text that has been normalised.

### Overview of the Model's Task
Receive tweets in json structure, where there is only tweet text:
```json
{"texts": [
    "T1 services delayed due to heavy rain near Strathfield.",
    "T1 Line services cancelled between Parramatta and Central due to flooding.",
    "T1 Line services cancelled due to flooding on tracks."
]}
```
Then the model will do through and classify each of the tweets to get a response that looks like:
```json
{"predictions": ["cancelled", "cancelled", "delayed"]}
```


---

## Steps:
1. Load the custom dataset
 2. Analyse the custom dataset for irregularities
 3. Clean the data (with the normalise function)
 4. Make the train/validation split
 5. Train the model on sagemaker
 6. Deploy the model with a serverless instance (allocating custom ram/concurrency)
 7. Test the model with our use cases


---


## Model of Choice
We chose an **SVM (support vector machine)** based model for our classification task as we have a small dataset of ~2000 tweets which we had labeled by hand to allow for the model to be trained. Given that we could not use a large model as it would very likely be overfitted to the small data we had.

### Models we considered
 | Model | Overview | Benefits | Drawbacks | Cost |
|------|---------|----------|-----------|------|
| Finetuned Distill-BERT | Smaller, faster version of BERT that retains approximately 97% of its performance | State-of-the-art accuracy | Higher inference cost compared to linear models | Training: Low<br>Inference: Medium |
| Support Vector Machines (via scikit-learn) | Classic margin-based classifier effective for high-dimensional sparse text data | Robust against overfitting with small datasets and extremely fast inference | Does not capture deep semantic meaning and requires manual feature engineering (TF-IDF) | Training: Very Low<br>Inference: Very Low |
| Multinomial Naive Bayes | Probabilistic classifier based on word frequency | Simplest baseline and handles limited data well by assuming feature independence | “Naive” independence assumption can reduce accuracy when word context is important | Training: Lowest<br>Inference: Lowest |
| Logistic Regression with Regularisation | Linear model using a softmax function for classification | Easy to interpret and L1/L2 regularisation helps prevent overfitting on small datasets | Struggles to model non-linear relationships in text | Training: Very Low<br>Inference: Very Low |
| SageMaker BlazingText | Built-in SageMaker algorithm optimised for text classification | Highly scalable and very fast training | Typically requires larger datasets to outperform pre-trained transformer models | Training: Low<br>Inference: Low–Medium |
| XGBoost | Gradient-boosted decision tree algorithm commonly used for structured and text data | High performance with built-in support for class balancing | Hyperparameter tuning can be time consuming for small datasets | Training: Low<br>Inference: Low |
| Random Forest Classifier | Ensemble of decision trees that reduces variance | Less prone to overfitting than a single decision tree and handles outliers well | Less effective at capturing linguistic patterns in text | Training: Very Low<br>Inference: Very Low |

---

### Final Notes

TODO: WILL WRITE SOMETHING LATER WHEN I HAVE THE FINAL THOUGHTS

## Initialisation

Loading the required libraries and getting the session, role and required elements from the aws sagemaker setup.

In [5]:
import os
import json
import time
import pandas as pd

import sagemaker
from sagemaker import get_execution_role
from sagemaker.session import Session
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.inputs import TrainingInput

In [6]:
sess = sagemaker.Session()
role = get_execution_role()
region = sess.boto_region_name
default_bucket = sess.default_bucket()

## Setting Up and Loading the Dataset

Loading the data set, showing its structure and understanding the small cleaning process we would need to undertake to normalise the text for the model to have an easier time during classification.

In [7]:
LOCAL_CSV_PATH = "sydney_trains_dc_dataset_2000.csv"
S3_CSV_URI = None
S3_PREFIX = "alert-service-mvp"

In [8]:
def load_dataset(local_path: str | None, s3_uri: str | None) -> pd.DataFrame:
    if s3_uri:
        import boto3
        from urllib.parse import urlparse

        parsed = urlparse(s3_uri)
        bucket = parsed.netloc
        key = parsed.path.lstrip("/")

        local_tmp = "/tmp/dataset.csv"
        boto3.client("s3").download_file(bucket, key, local_tmp)
        return pd.read_csv(local_tmp)

    if not local_path:
        raise ValueError("Provide either LOCAL_CSV_PATH or S3_CSV_URI")

    return pd.read_csv(local_path)

df = load_dataset(LOCAL_CSV_PATH, S3_CSV_URI)

In [9]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (2000, 2)
Columns: ['text', 'classification']


In [10]:
display(df.head(10))

,text,classification
0,#WesternLine Allow up to 10 minutes of extra t...,Delayed
1,#NorthShoreLine & #WesternLine Allow extra tra...,Delayed
2,#WesternLine Allow extra travel time from the ...,Delayed
3,#WesternLine Allow extra travel time away from...,Delayed
4,#WesternLine Allow up to 15 minutes of extra t...,Delayed
5,#NorthShoreLine #WesternLine Trains are not ru...,Cancelled
6,#WesternLine #NorthShoreLine Allow up to 10 mi...,Delayed
7,#WesternLine Allow up to 10 minutes of extra t...,Delayed
8,#NorthShoreLine & #WesternLine Allow up to 15 ...,Delayed
9,#NorthShoreLine #WesternLine Trains are not ru...,Cancelled


### Cleaning the dataset

Using the normalize_text function to ensure that the text the model is going to receive will be:
- lowercase
- without any URLs
- without any mentions
- converted hashtags as to not lose the meaning of the text but removing characters with special meaning not relevant to the task at hand (#flooding -> flooding)
- without any emojis / non-ascii characters
- without any punctuation

In [11]:
import re

required = {"text", "classification"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Drop rows with missing fields
df = df.dropna(subset=["text", "classification"]).copy()

## NOTE this function is useful as this will be the cleanup steps we will need to do before we send the model our live feed data ##
def normalize_text(text: str) -> str:
    """
    Normalises tweet text to reduce noise for TF-IDF features.
    Steps:
    - lowercase
    - remove URLs
    - remove mentions
    - convert hashtags (#flooding -> flooding)
    - remove emojis / non-ascii
    - remove punctuation
    - collapse whitespace
    """

    text = text.lower()

    # remove urls
    text = re.sub(r"http\S+|www\S+", "", text)

    # remove mentions
    text = re.sub(r"@\w+", "", text)

    # convert hashtags to plain words
    text = re.sub(r"#(\w+)", r"\1", text)

    # remove emojis / non ascii
    text = text.encode("ascii", "ignore").decode()

    # remove punctuation
    text = re.sub(r"[^\w\s]", " ", text)

    # collapse extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


df["text"] = df["text"].astype(str).apply(normalize_text)
df["classification"] = df["classification"].astype(str).str.strip().str.lower()

allowed = {"delayed", "cancelled"}
df = df[df["classification"].isin(allowed)].copy()

In [12]:
# Post Cleaning

print("\nClass distribution:")
print(df["classification"].value_counts())

print("\nSample cleaned tweets:")
display(df.head(10))


Class distribution:
classification
delayed      1692
cancelled     300
Name: count, dtype: int64

Sample cleaned tweets:


,text,classification
0,westernline allow up to 10 minutes of extra tr...,delayed
1,northshoreline westernline allow extra travel ...,delayed
2,westernline allow extra travel time from the c...,delayed
3,westernline allow extra travel time away from ...,delayed
4,westernline allow up to 15 minutes of extra tr...,delayed
5,northshoreline westernline trains are not runn...,cancelled
6,westernline northshoreline allow up to 10 minu...,delayed
7,westernline allow up to 10 minutes of extra tr...,delayed
8,northshoreline westernline allow up to 15 minu...,delayed
9,northshoreline westernline trains are not runn...,cancelled


## Setting up the Train/Test Split

Chose the test size of 30% of the dataset size as we really wanted to avoid the model overfitting and underperforming on validation.

In [13]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=df["classification"]
)

os.makedirs("data", exist_ok=True)
train_path = "data/train.csv"
val_path = "data/val.csv"

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)

In [14]:
print("Train:", train_df.shape, "Val:", val_df.shape)

Train: (1394, 2) Val: (598, 2)


In [15]:
train_s3 = sess.upload_data(train_path, bucket=default_bucket, key_prefix=f"{S3_PREFIX}/train")
val_s3 = sess.upload_data(val_path, bucket=default_bucket, key_prefix=f"{S3_PREFIX}/val")

## Training the Model

In training the model we set up the training instance-type, load the `train.py` file which is the master script how to train the model and show where it is stored. There is also some setup work with using the instance type and framework versions of scikit-learn and python we are using.

Then the `fit` function is called where we will train the model and initialise the training job as we have detailed in `train.py`

In [ ]:
sk_estimator = SKLearn(
    entry_point="train.py",
    source_dir="src",
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    framework_version="1.2-1",
    py_version="py3"
)

sk_estimator.fit(
    {
        "train": train_s3,
        "val": val_s3
    }
)

## Deploy the Model

The deployment instance type we chose for this model was the **Sagemaker Serverless Inference Config**. This was as it was the most budget friendly option and had the least amount of setup involved.

The memory size we used in this case is 2 GB and a max concurrency setting of 5

In [17]:
from sagemaker.serverless import ServerlessInferenceConfig

serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=2048,
    max_concurrency=5
)

In [18]:
predictor = sk_estimator.deploy(
    serverless_inference_config=serverless_config,
    entry_point="inference.py",
    source_dir="src"
)

print("Serverless endpoint:", predictor.endpoint_name)

INFO:sagemaker:Creating model with name: sagemaker-scikit-learn-2026-03-10-02-52-39-870
INFO:sagemaker:Creating endpoint-config with name sagemaker-scikit-learn-2026-03-10-02-52-39-870
INFO:sagemaker:Creating endpoint with name sagemaker-scikit-learn-2026-03-10-02-52-39-870


------!Serverless endpoint: sagemaker-scikit-learn-2026-03-10-02-52-39-870


In [19]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

Light manual testing to see the model is working as expected

In [21]:
# Single
print(predictor.predict({"text": "T1 Line services cancelled due to flooding on tracks."}))

# Batch
payload = {"texts": [
    "T1 services delayed due to heavy rain near Strathfield.",
    "T1 Line services cancelled between Parramatta and Central due to flooding.",
    "T1 Line services cancelled due to flooding on tracks."
]}
print(predictor.predict(payload))

{'prediction': 'delayed'}
{'predictions': ['cancelled', 'cancelled', 'delayed']}


In [ ]:
predictor.delete_endpoint()